In [ ]:
import hashlib
import matplotlib.pyplot as plb
import numpy

## 前端
### 計算專屬的 Commitment

In [ ]:
def calculate_commitment(student_id, secret):
    combined = f"{student_id}:{secret}".encode('utf-8')
    return hashlib.sha256(combined).hexdigest()

### 產生 ZKP Proof 
(還要修改)

In [ ]:
def generate_simulated_proof(student_id, secret):
    proof_data = f"proof_{student_id}_{secret}".encode('utf-8')
    return hashlib.sha256(proof_data).hexdigest()

In [ ]:
class FlawedBackendDB:
    def __init__(self):
        # 註冊表：記錄「學號」對應哪個「Commitment」(這就是破綻)
        self.registrations = {} 
        # 投票紀錄：記錄哪些 Commitment 已經投過票
        self.has_voted = set()
        self.votes = []

    def register(self, student_id, commitment):
        print(f"[後端日誌] 收到註冊請求 - 學號: {student_id}, Commitment: {commitment[:8]}...")
        self.registrations[student_id] = commitment

    def submit_vote(self, commitment, proof, encrypted_vote):
        print(f"[後端日誌] 收到投票請求 - 驗證 Commitment: {commitment[:8]}...")
        
        # 防線一：檢查是否合法註冊過
        if commitment not in self.registrations.values():
            return "❌ 拒絕：此 Commitment 未註冊"
            
        # 防線二：檢查是否重複投票
        if commitment in self.has_voted:
            return "❌ 拒絕：此人已投過票"

        # (這裡省略 ZKP 的數學驗證，假設 Proof 驗證通過)

        # 寫入選票並註記已投票
        self.votes.append(encrypted_vote)
        self.has_voted.add(commitment)
        return "✅ 投票成功，選票已存入票匭"

    def hacker_perspective(self, target_commitment):
        """模擬後端工程師 (內鬼) 的視角：利用資料庫反查投票者身分"""
        print("\n🚨 [內鬼駭客視角] 發現新選票，開始反查資料庫...")
        for student_id, reg_commitment in self.registrations.items():
            if reg_commitment == target_commitment:
                print(f"💀 破解成功！剛剛投下這張票的 Commitment ({target_commitment[:8]}...) 屬於學號: {student_id}")
                return student_id
        return "未找到關聯"

In [ ]:
if __name__ == "__main__":
    # 1. 初始化環境
    db = FlawedBackendDB()
    student_id = "1120001"
    secret = "my_super_secret_password"

    # 2. 註冊階段 (對應你前端的 initKeyAndRegister)
    print("--- 註冊階段 ---")
    my_commitment = calculate_commitment(student_id, secret)
    db.register(student_id, my_commitment)

    # 3. 投票階段 (對應你前端的 handleVote)
    print("\n--- 投票階段 ---")
    my_proof = generate_simulated_proof(student_id, secret)
    vote_result = db.submit_vote(
        commitment=my_commitment, 
        proof=my_proof, 
        encrypted_vote="[RSA加密選票: 投給候選人A]"
    )
    print(vote_result)

    # 4. 見光死瞬間 (後端反查)
    # 駭客(後端工程師)直接拿剛剛收到的 Commitment 去查字典
    db.hacker_perspective(my_commitment)